### 1. 패키지 설치

In [1]:
%pip install -qU langchain langchain-openai langchain-unstructured "unstructured[docx]" langchain-chroma python-dotenv

Note: you may need to restart the kernel to use updated packages.


### 2. 환경변수 로드

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

### 3. 파싱 + 청킹

In [3]:
from langchain_unstructured import UnstructuredLoader

loader = UnstructuredLoader(
    file_path='../../files/doc/tax.docx',
    chunking_strategy='by_title',
    max_characters=1000,
    overlap=200,
    include_orig_elements=False,
    languages=['kor', 'eng']
)

docs = loader.load()

In [4]:
len(docs)

275

### 4. 임베딩(설정만)
- 여기선 임베딩 객체만 생성, 실제 벡터화는 5번에서 벡터 스토어가 내부 호출함

In [5]:
from langchain_openai import OpenAIEmbeddings

embedding = OpenAIEmbeddings(
    model='text-embedding-3-large'
)

# vectors = embedding.embed_documents([doc.page_content for doc in docs])

### 5. 벡터 스토어
- 방법1: 벡터 스토어 생성 → 문서 추가

In [6]:
from langchain_chroma import Chroma

vector_store1 = Chroma(
    collection_name='tax_docx_1',
    embedding_function=embedding, # embedding.embed_documents로 벡터화 하는 과정은 chroma에서 알아서 호출해서 사용함
    persist_directory='./chroma-db' # 영속화
)

In [7]:
from uuid import uuid4

uuids = [str(uuid4()) for _ in range(len(docs))]
vector_store1.add_documents(documents=docs, ids=uuids)

INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


['baf61b89-37a9-452b-9e81-f77b802de547',
 '45f838ba-7ab6-42fa-9494-d7b45e854b52',
 '28b518e8-1740-4c40-b370-606d43d74f6b',
 '0e091608-20f8-4f10-99d9-4535effedac2',
 '0201f966-23da-406d-8418-c5816d3ff8c1',
 'dc9d8676-f7ad-4048-b845-1d9628c72356',
 '622894fb-1098-4172-9eaf-d8609fea3d34',
 '830e6989-834f-4500-a8b0-198005c609c1',
 '9603d412-f6ba-44d6-8f07-dceaf025a216',
 '14dc653e-cea6-4e7f-9ede-928e57c52340',
 'aa4d5e06-7f71-484f-a2bc-65552a6fdce3',
 'b9c92a6d-7729-427f-9038-dc1a41cd35c0',
 '8fa476ed-952b-4c8d-9483-b0ed31256fa7',
 '4d994bf6-1770-457d-a426-861b820725b3',
 'd83d2f8c-a5f9-47b0-a3cb-0b3804c8fe1a',
 '23d32991-ba24-41cd-851c-f294d4754cb5',
 '98395ac9-4d3e-44c5-a79e-ae062f8ec9f7',
 '9ba4eba9-561c-49a5-be28-3de1051e077c',
 '86ef84f7-1065-4e2c-a92a-bb738a8fdc6c',
 '3197e641-0661-4cf2-bcd2-f995c2fcd523',
 '9841d7f2-ab6e-428f-a275-cb130b038957',
 '127866c5-3850-4968-982a-fcbcde750a12',
 'a829a69b-d0aa-4792-a863-d0ab81c2e043',
 '3c3a2e10-11bb-4823-9532-ead4a4f82279',
 'e1560148-dc5a-

- 방법2: 벡터 스토어 생성 + 문서 추가

In [8]:
from langchain_chroma import Chroma

vector_store2 = Chroma.from_documents(
    documents=docs,
    embedding=embedding,
    collection_name='tax_docx_2',
    persist_directory='./chroma-db_2' # 영속화
)

INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


### 6. 수정 (update)
- id값으로 찾아 새로운 문서 내용으로 변경

In [9]:
docs[0].metadata
#docs[0].metadata['source']
#docs[0].page_content

#vector_store1.get()
#vector_store1.get(limit=3)
#vector_store1._collection_name

{'source': '../../files/doc/tax.docx',
 'emphasized_text_contents': ['소득세법',
  '제',
  '1',
  '장',
  '총칙',
  '제',
  '1',
  '조',
  '(',
  '목적',
  ')',
  '제',
  '1',
  '조의',
  '2(',
  '정의',
  ')',
  '제',
  '2',
  '조',
  '(',
  '납세',
  '의무',
  ')'],
 'emphasized_text_tags': ['b',
  'b',
  'b',
  'b',
  'b',
  'b',
  'b',
  'b',
  'b',
  'b',
  'b',
  'b',
  'b',
  'b',
  'b',
  'b',
  'b',
  'b',
  'b',
  'b',
  'b',
  'b',
  'b',
  'b'],
 'file_directory': '../../files/doc',
 'filename': 'tax.docx',
 'last_modified': '2026-06-08T20:44:05',
 'page_number': 1,
 'languages': ['kor', 'eng'],
 'filetype': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document',
 'category': 'CompositeElement',
 'element_id': 'd765929abbe47629f251ff5cabdec0fc'}

In [10]:
from langchain_core.documents import Document

updated_doc = Document(
    page_content='수정 내용', # 특정 docs[번호].page_content 값 변경
    #metadata={'source': '../../files/doc/tax.docx'} # 특정 docs[번호].metadata 값 똑같이 입력
    metadata={'source': docs[0].metadata['source']} # 특정 docs[번호].metadata 값 똑같이 입력
)

vector_store1.update_document(
    document_id=uuids[0], # 특정 uuid값 선택
    document=updated_doc # 변경할 내용으로 수정
)

vector_store1.get(ids=[uuids[0]]) # 변경된 내용 확인

INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


{'ids': ['baf61b89-37a9-452b-9e81-f77b802de547'],
 'embeddings': None,
 'documents': ['수정 내용'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'element_id': 'd765929abbe47629f251ff5cabdec0fc',
   'emphasized_text_contents': ['소득세법',
    '제',
    '1',
    '장',
    '총칙',
    '제',
    '1',
    '조',
    '(',
    '목적',
    ')',
    '제',
    '1',
    '조의',
    '2(',
    '정의',
    ')',
    '제',
    '2',
    '조',
    '(',
    '납세',
    '의무',
    ')'],
   'file_directory': '../../files/doc',
   'filetype': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document',
   'source': '../../files/doc/tax.docx',
   'emphasized_text_tags': ['b',
    'b',
    'b',
    'b',
    'b',
    'b',
    'b',
    'b',
    'b',
    'b',
    'b',
    'b',
    'b',
    'b',
    'b',
    'b',
    'b',
    'b',
    'b',
    'b',
    'b',
    'b',
    'b',
    'b'],
   'filename': 'tax.docx',
   'languages': ['kor', 'eng'],
   'page_number': 1,
   'last_modified': 

### 7. 삭제 (delete)
- id값으로 찾아 문서 제거

In [11]:
vector_store1.delete(
    ids=[uuids[0]]
)

vector_store1.get(ids=[uuids[0]])

{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': []}